In [27]:
from pyspark.sql import SparkSession
buildings_path = "Buildings.csv"
energyConsum_path = "EnergyConsumptionForecast.csv"
historicEnergyConsum_path = "HistoricalEnergyConsumption.csv"

def calculate_abs_value(current, hist):
    return (abs(current - hist) / hist * 100)
    
session = SparkSession.builder.getOrCreate()

buildings = session.read.load(buildings_path, format = "csv", header = False, inferSchema = True)
energyConsum = session.read.load(energyConsum_path, format = "csv", header = False, inferSchema = True)
historConsum = session.read.load(historicEnergyConsum_path, format = "csv", header = False, inferSchema = True)

historConsumCleaned = historConsum.filter("_c2 != 0")

energyConsum = energyConsum.selectExpr("_c0 as bID", "_c1 as date", "_c2 as currConsum")
historConsumCleaned = historConsumCleaned.selectExpr("_c0 as ID", "_c1 as date1", "_c2 as histConsum")

#Join the historical data with the current data
joinedDF = energyConsum.join(historConsumCleaned, on= ((energyConsum["bID"] == historConsumCleaned["ID"]) & (energyConsum["date"] == historConsumCleaned["date1"])))
session.udf.register("calculate_abs", calculate_abs_value)

joinedDF.createOrReplaceTempView("joinedEnergyConsum")

absValueDF = session.sql("select bID, date, calculate_abs(currConsum, histConsum) as AbsConsum from joinedEnergyConsum  ")
filteredDF = absValueDF.filter("AbsConsum >= 20.0")

buildings = buildings.selectExpr("_c6 as Class", "_c4 as type", "_c0 as BuildID")

buildings = buildings.join(filteredDF, on = filteredDF["bID"] == buildings["BuildID"])

buildings.createOrReplaceTempView("TypesWithAbsConsum")

finalDF = session.sql("select class, type, count(AbsConsum) from TypesWithAbsConsum group by class, type")

buildings.show()
finalDF.show()

25/12/18 17:21:45 WARN SimpleFunctionRegistry: The function calculate_abs replaced a previously registered function.


+-----+-----------+-------+------+-------------+-----------------+
|Class|       type|BuildID|   bID|         date|        AbsConsum|
+-----+-----------+-------+------+-------------+-----------------+
|    C|Residential| B_1001|B_1001|2024/03/15-14|66.44736842105263|
|    C|Residential| B_1001|B_1001|2024/03/15-16|63.69426751592358|
|    B| Industrial| B_1004|B_1004|2024/03/15-18|67.22222222222223|
+-----+-----------+-------+------+-------------+-----------------+

+-----+-----------+----------------+
|class|       type|count(AbsConsum)|
+-----+-----------+----------------+
|    C|Residential|               2|
|    B| Industrial|               1|
+-----+-----------+----------------+

